# Homework 3
Machine Learning for Classification

[Homework Link](https://github.com/alexeygrigorev/mlbookcamp-code/blob/master/course-zoomcamp/cohorts/2022/03-classification/homework.md) 

## Data Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
col_names = [
'latitude',
'longitude',
'housing_median_age',
'total_rooms',
'total_bedrooms',
'population',
'households',
'median_income',
'median_house_value',
'ocean_proximity']

In [3]:
df = pd.read_csv('housing.csv')
df = df[col_names]

## Data Preparation


In [4]:
df['rooms_per_house_hold'] = df.total_rooms/df.households
df['bedrooms_per_room'] = df.total_bedrooms/df.total_rooms
df['population_per_household'] = df.population/df.households

In [5]:
# Convenience method to refresh df incase it was modified
def refresh_data():
    df = pd.read_csv('housing.csv')
    df = df[col_names]
    df.fillna(0, inplace=True)
    # addnew rows
    df['rooms_per_house_hold'] = df.total_rooms/df.households
    df['bedrooms_per_room'] = df.total_bedrooms/df.total_rooms
    df['population_per_household'] = df.population/df.households
    
    return df

In [6]:
df = refresh_data()

## Split the Data


60/20/20
using sklearn train_test_split, set seed to 42

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
# Convenience method for uniform data preparation
def split_data(dataframe):
    df = dataframe.copy()
    
    # split data
    df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
    
    # reset index
    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)
    
    # y
    y_train = df_train.median_house_value.values
    y_val = df_val.median_house_value.values
    y_test = df_test.median_house_value.values
    
    # remove from dfs
    del df_train['median_house_value']
    del df_val['median_house_value']
    del df_test['median_house_value']
    
    # return a dict
    data = {
        'df_train': df_train,
        'df_val': df_val,
        'df_test': df_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test
    }
    
    return data

In [9]:
# Helper method to prepare train and val data
from sklearn.feature_extraction import DictVectorizer
def prepare_X(df_train, df_val):
    dv = DictVectorizer(sparse=False)
    
    dict_train = df_train[categorical+numerical].to_dict(orient='records')
    dv.fit(dict_train)
    X_train = dv.transform(dict_train)

    #val
    dict_val = df_val[categorical+numerical].to_dict(orient='records')
    X_val = dv.transform(dict_val)
    
    return X_train, X_val


In [10]:
data = split_data(refresh_data())

## Question 1


In [11]:
df.ocean_proximity.value_counts()

<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: ocean_proximity, dtype: int64

In [12]:
df.ocean_proximity.mode()


0    <1H OCEAN
Name: ocean_proximity, dtype: object

Q1. B. <1H Ocean

## Question 2

In [13]:
df = refresh_data()

In [14]:
data = split_data(df)

In [15]:
df_train = data['df_train']

In [16]:
df_train.columns

Index(['latitude', 'longitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'ocean_proximity', 'rooms_per_house_hold', 'bedrooms_per_room',
       'population_per_household'],
      dtype='object')

In [17]:
df_train.dtypes

latitude                    float64
longitude                   float64
housing_median_age          float64
total_rooms                 float64
total_bedrooms              float64
population                  float64
households                  float64
median_income               float64
ocean_proximity              object
rooms_per_house_hold        float64
bedrooms_per_room           float64
population_per_household    float64
dtype: object

In [18]:
numerical = ['latitude', 'longitude', 'housing_median_age', 
             'total_rooms', 'total_bedrooms', 
             'population', 'households', 'median_income',
             'rooms_per_house_hold', 'bedrooms_per_room',
             'population_per_household']

In [19]:
df_numerical = df_train[numerical]
df_cor = df_numerical.corr()

In [20]:
df_cor[df_cor != 1]

,latitude,longitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,rooms_per_house_hold,bedrooms_per_room,population_per_household
latitude,NaN,-0.925005,0.002477,-0.025914,-0.059730,-0.100272,-0.063529,-0.076805,0.119118,-0.124507,-0.002301
longitude,-0.925005,NaN,-0.099812,0.036449,0.063840,0.091670,0.049762,-0.016426,-0.034814,0.102320,0.011022
housing_median_age,0.002477,-0.099812,NaN,-0.363522,-0.324156,-0.292476,-0.306119,-0.119591,-0.181275,0.129456,0.012167
total_rooms,-0.025914,0.036449,-0.363522,NaN,0.931546,0.853219,0.921441,0.198951,0.168926,-0.194185,-0.029452
total_bedrooms,-0.059730,0.063840,-0.324156,0.931546,NaN,0.877340,0.979399,-0.009833,0.010381,0.078094,-0.034301
population,-0.100272,0.091670,-0.292476,0.853219,0.877340,NaN,0.906841,-0.000849,-0.076210,0.031592,0.064998
households,-0.063529,0.049762,-0.306119,0.921441,0.979399,0.906841,NaN,0.011925,-0.085832,0.058004,-0.032522
median_income,-0.076805,-0.016426,-0.119591,0.198951,-0.009833,-0.000849,0.011925,NaN,0.394154,-0.616617,-0.000454
rooms_per_house_hold,0.119118,-0.034814,-0.181275,0.168926,0.010381,-0.076210,-0.085832,0.394154,NaN,-0.500589,0.001801
bedrooms_per_room,-0.124507,0.102320,0.129456,-0.194185,0.078094,0.031592,0.058004,-0.616617,-0.500589,NaN,-0.002851


In [21]:
df_cor[df_cor != 1].max()

latitude                    0.119118
longitude                   0.102320
housing_median_age          0.129456
total_rooms                 0.931546
total_bedrooms              0.979399
population                  0.906841
households                  0.979399
median_income               0.394154
rooms_per_house_hold        0.394154
bedrooms_per_room           0.129456
population_per_household    0.064998
dtype: float64

Q2. A. total_bedrooms and households

## Question 3

In [22]:
y_train = data['y_train'] # median house value for train 

In [23]:
y_train

array([241400., 500001.,  64100., ..., 215300., 139000., 181300.])

In [24]:
above_average = (y_train > y_train.mean()).astype(int)
above_average

array([1, 1, 0, ..., 1, 0, 0])

In [25]:
df_train.dtypes

latitude                    float64
longitude                   float64
housing_median_age          float64
total_rooms                 float64
total_bedrooms              float64
population                  float64
households                  float64
median_income               float64
ocean_proximity              object
rooms_per_house_hold        float64
bedrooms_per_room           float64
population_per_household    float64
dtype: object

In [26]:
from sklearn.metrics import mutual_info_score

In [27]:
mutual_info_score(df_train.ocean_proximity, above_average)

0.10138385763624205

Q3. C. 0.101

## Question 4

In [28]:
categorical = ['ocean_proximity']

In [29]:
numerical = ['latitude', 'longitude', 'housing_median_age', 
             'total_rooms', 'total_bedrooms', 
             'population', 'households', 'median_income',
             'rooms_per_house_hold', 'bedrooms_per_room',
             'population_per_household']

In [30]:
X_train, X_val = prepare_X(data['df_train'], data['df_val'])


In [31]:
y_train = data['y_train']
y_train = (y_train > y_train.mean()).astype(int)

In [32]:
y_val = data['y_val']
y_val = (y_val > y_val.mean()).astype(int)

In [33]:
from sklearn.linear_model import LogisticRegression


In [34]:
model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)

In [35]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [36]:
# w0
w0 = model.intercept_[0]
w0

-0.0711147642719202

In [37]:
w = model.coef_[0]
w

array([ 1.43247685e-01,  3.93458265e-03,  3.57935853e-02,  1.02961719e-01,
        8.29418807e-02,  1.21086557e+00,  4.57298652e-01, -1.61253193e+00,
        1.49769900e-02,  2.92521794e-01,  7.76619726e-01, -1.63353330e-03,
        9.91393539e-03, -2.12150953e-02,  1.91158597e-03, -1.53394085e-04])

In [38]:
y_pred = model.predict_proba(X_val)[:, 1]

In [39]:
y_decision = (y_pred > 0.5).astype(int)

In [40]:
global_accuracy = (y_val == y_decision).mean()

In [41]:
round(global_accuracy, 2)

0.84

Q4. C. 0.84

## Question 5. Feature Elimination

In [42]:
global_accuracy

0.8369670542635659

In [43]:
df_train.columns

Index(['latitude', 'longitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'ocean_proximity', 'rooms_per_house_hold', 'bedrooms_per_room',
       'population_per_household'],
      dtype='object')

In [44]:
numerical_alt = [
 'latitude', 'longitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income']

In [45]:
numerical = ['latitude', 'longitude','housing_median_age', 
             'total_rooms', 'total_bedrooms', 
             'population', 'households', 'median_income',
             'rooms_per_house_hold', 'bedrooms_per_room',
             'population_per_household']

In [64]:
numerical_main = numerical.copy()

In [65]:
categorical

['ocean_proximity']

In [66]:
scores = list()
min_diff = (None, None, float('inf'))
df = refresh_data()
for i in numerical_main:
    numerical = numerical_main.copy()
    numerical.remove(i)
    
    data = split_data(df)

    dv = DictVectorizer(sparse=False)

    #X
    X_train, X_val = prepare_X(data['df_train'], data['df_val'])
    
    # Y train
    y_train = data['y_train']
    y_train = (y_train > y_train.mean()).astype(int)

    # Y val
    y_val = data['y_val']
    y_val = (y_val > y_val.mean()).astype(int)


    #model
    model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred=model.predict_proba(X_val)[:, 1]
    y_decision = (y_pred > 0.5).astype(int)

    acc = (y_val == y_decision).mean()
    diff = abs(global_accuracy-acc)
    scores.append((i, acc, diff))
    
    if diff < min_diff[2]:
        min_diff = (i, acc, diff)


In [67]:
scores

[('housing_median_age', 0.8333333333333334, 0.003633720930232509),
 ('total_rooms', 0.8374515503875969, 0.0004844961240310086),
 ('total_bedrooms', 0.8359980620155039, 0.0009689922480620172),
 ('population', 0.8309108527131783, 0.006056201550387552),
 ('households', 0.8374515503875969, 0.0004844961240310086),
 ('median_income', 0.7877906976744186, 0.04917635658914732),
 ('rooms_per_house_hold', 0.8357558139534884, 0.001211240310077466),
 ('bedrooms_per_room', 0.8379360465116279, 0.0009689922480620172),
 ('population_per_household', 0.8374515503875969, 0.0004844961240310086)]

In [68]:
min_diff

('total_rooms', 0.8374515503875969, 0.0004844961240310086)

Q5. A. total_rooms

## QUESTION 6


In [69]:
from sklearn.linear_model import Ridge

In [70]:
# rmse method
def rmse(y, y_pred):
    se = (y - y_pred)** 2
    mse = se.mean()
    return np.sqrt(mse)

In [71]:
numerical = ['housing_median_age', 'total_rooms', 'total_bedrooms', 
             'population', 'households', 'median_income',
             'rooms_per_house_hold', 'bedrooms_per_room',
             'population_per_household']

In [72]:
categorical = ['ocean_proximity']

In [73]:
data = split_data(refresh_data())

In [74]:
X_train, X_val = prepare_X(data['df_train'], data['df_val'])


In [75]:
X_train

array([[2.59713701e-01, 3.74000000e+02, 3.90000000e+01, ...,
        3.92245989e+00, 3.81000000e+02, 1.46700000e+03],
       [1.30227981e-01, 8.06000000e+02, 2.40000000e+01, ...,
        7.56451613e+00, 7.94000000e+02, 6.09700000e+03],
       [2.34624146e-01, 3.37000000e+02, 4.10000000e+01, ...,
        3.90801187e+00, 3.09000000e+02, 1.31700000e+03],
       ...,
       [1.82879377e-01, 6.02000000e+02, 1.80000000e+01, ...,
        5.54983389e+00, 6.11000000e+02, 3.34100000e+03],
       [2.29126214e-01, 3.50000000e+02, 1.60000000e+01, ...,
        4.41428571e+00, 3.54000000e+02, 1.54500000e+03],
       [2.09574468e-01, 2.15000000e+02, 3.50000000e+01, ...,
        4.37209302e+00, 1.97000000e+02, 9.40000000e+02]])

In [76]:
scores = list()
min_score = (None, float('inf'))
df = refresh_data()
for a in [0, 0.01, 0.1, 1, 10]:
    model = Ridge(alpha=a, solver="sag", random_state=42)

    data = split_data(df)
    X_train, X_val = prepare_X(data['df_train'], data['df_val'])
    
    #Y train
    y_train = data['y_train']
    y_train = np.log1p(y_train)

    #Y val
    y_val = data['y_val']
    y_val = np.log1p(y_val)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    score = rmse(y_pred, y_val)
    
    scores.append((a, score))
    
    if score < min_score[1]:
        min_score = (a, score)
        
        

In [77]:
scores

[(0, 0.5249039141832536),
 (0.01, 0.5249039141984874),
 (0.1, 0.5249039143432098),
 (1, 0.5249039157980475),
 (10, 0.5249039303235633)]

In [78]:
min_score

(0, 0.5249039141832536)

Q6. A. 0